# Семинар NumPy (2 пары)

## Как работать с этим ноутбуком
1. Впиши свой идентификатор (почтовый логин) в переменную `STUDENT_ID`.
2. Выполняй задания по порядку. Для каждого задания:
   - реализуй код **без циклов Python** (если явно не разрешено),
   - сформируй требуемые переменные и выведи контрольную сумму,
   - кратко ответь на вопросы в Markdown (там, где требуется).
3. Итог сдачи: выполненный ноутбук `.ipynb` с сохраненными выводами.


## Источники
- Документация Python: https://www.python.org
- Документация NumPy: https://numpy.org/doc/stable/


In [2]:
import numpy as np
import hashlib, zlib
from pathlib import Path

# === 0) Впиши свой идентификатор (строка) ===
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"

def seed_from_student_id(student_id: str) -> int:
    # Детерминированный seed из строки (первые 4 байта SHA-256)
    h = hashlib.sha256(student_id.encode("utf-8")).digest()
    return int.from_bytes(h[:4], "little", signed=False)

BASE_SEED = seed_from_student_id(STUDENT_ID)

def task_rng(task_no: int) -> np.random.Generator:
    # Отдельный генератор для каждого задания (чтобы порядок выполнения не влиял на данные)
    return np.random.default_rng(BASE_SEED + task_no * 1000 + 12345)

def checksum(obj) -> int:
    # CRC32 для ndarray/bytes/строк/чисел — чтобы фиксировать ответ
    if isinstance(obj, np.ndarray):
        b = obj.tobytes()
    elif isinstance(obj, (bytes, bytearray)):
        b = bytes(obj)
    else:
        b = str(obj).encode("utf-8")
    return zlib.crc32(b) & 0xffffffff

np.set_printoptions(precision=4, suppress=True)
print("BASE_SEED =", BASE_SEED)


BASE_SEED = 4020147853


## Задание 01 — Представления (view) vs копии (copy) и память
1) Сгенерируй одномерный массив `a` длиной `n` из целых чисел (см. код ниже).
2) Получи **представление** `b = a.reshape(r, c)` (reshape должен быть без копирования).
3) Получи **представление** `c = b.T` (транспонирование).
4) Измени **ровно 5** элементов в `c` (индексы вычисли из `rng`, см. ниже) так, чтобы изменения отразились в `a`.
5) Выведи:
   - `np.shares_memory(a, b)` и `np.shares_memory(a, c)`,
   - флаги `a.flags["OWNDATA"]`, `b.flags["OWNDATA"]`, `c.flags["OWNDATA"]`,
   - контрольную сумму `checksum(a)` после изменений.

Требования: не использовать циклы `for/while` (можно векторную индексацию).


In [3]:
rng = task_rng(1)
n = int(rng.integers(48, 81))         # длина 1D массива
r = int(rng.integers(6, 10))          # строки
c = n // r                            # столбцы
n = r * c

a = rng.integers(-50, 51, size=n, dtype=np.int32)

# TODO: b, c, изменить 5 элементов в c
# YOUR CODE HERE

# TODO: выводы и checksum(a)
# YOUR CODE HERE


## Задание 02 — dtype, память и ошибки округления
1) Сгенерируй матрицу `x64` формы `(m, n)` из `float64` в диапазоне `[0, 1)`.
2) Получи варианты: `x32 = x64.astype(np.float32)`, `x16 = x64.astype(np.float16)`.
3) Посчитай:
   - объем памяти `nbytes` для каждого массива,
   - максимальную абсолютную ошибку `max(|x64 - x32|)` и `max(|x64 - x16|)`.
4) Ответь в Markdown: почему при уменьшении точности растет ошибка?

Сохрани результаты:
`mem_02 = (x64.nbytes, x32.nbytes, x16.nbytes)`,
`err_02 = (err32, err16)`.


In [ ]:
rng = task_rng(2)
m = int(rng.integers(120, 181))
n = int(rng.integers(60, 91))

# YOUR CODE HERE


## Задание 03 — Нормализация аффинным преобразованием ax+b
Сгенерируй массив `arr` формы `(4, 7)` из равномерного распределения на `[0, 20)`.
Нормализуй `arr` преобразованием `a*x + b`, чтобы после нормализации:
- `arr_norm.min() == 0.0`,
- `arr_norm.max() == 1.0`.

Дополнительно:
- проверь, что нормализация корректно работает даже если `min == max` (обработай этот случай),
- выведи `checksum(arr_norm)`.

Сохрани `arr_norm` в `ans_03`.


In [ ]:
rng = task_rng(3)
arr = rng.random((4, 7)) * 20

# YOUR CODE HERE


## Задание 04 — Строка с минимальной суммой + детерминированный tie-break
1) Создай матрицу `M` формы `(8, 10)` из целых 0..10.
2) Найди индекс строки с **минимальной суммой**.
3) Если таких строк несколько, выбери среди них строку с **максимальной дисперсией**.
4) Выведи: индекс, строку, сумму и дисперсию. Контрольная сумма строки: `checksum(row)`.

Сохрани:
- `idx_04` (int),
- `row_04` (ndarray shape (10,)).


In [ ]:
rng = task_rng(4)
M = rng.integers(0, 11, size=(8, 10), dtype=np.int32)

# YOUR CODE HERE


## Задание 05 — Сохранение/загрузка npz и оценка «коэффициента сжатия»
1) Сгенерируй два массива:
   - `A` формы `(200, 50)` float64,
   - `B` формы `(200,)` int32.
2) Сохрани их в **сжатый** архив `data_05.npz` с ключами `"A"` и `"B"`.
3) Загрузи файл обратно и проверь равенство данных.
4) Посчитай:
   - размер файла на диске (байт),
   - размер «сырых данных» (`A.nbytes + B.nbytes`),
   - коэффициент сжатия = raw / file_size.

Сохрани:
`ratio_05` (float), `ok_05` (bool).


In [ ]:
rng = task_rng(5)
A = rng.standard_normal((200, 50))
B = rng.integers(-10_000, 10_001, size=200, dtype=np.int32)

# YOUR CODE HERE


## Задание 06 — Срезы + broadcasting: «шахматная» модификация
1) Создай матрицу `X` формы `(12, 16)` из целых 0..99.
2) Без циклов:
   - все элементы на позициях с четной строкой и четным столбцом увеличь на 1000,
   - все элементы на позициях с нечетной строкой и нечетным столбцом уменьшить на 100.
3) Проверь, что изменилось ровно `X.size/2` элементов и выведи `checksum(X)`.

Сохрани `ans_06 = X`.


In [ ]:
rng = task_rng(6)
X = rng.integers(0, 100, size=(12, 16), dtype=np.int32)

# YOUR CODE HERE


## Задание 07 — Комбинация ufunc: стабильные вычисления
Сгенерируй массивы `P` и `Q` формы `(1000,)`:
- `P` ~ N(0, 1),
- `Q` ~ U(0, 1).

Вычисли вектор:
`R = log1p(exp(P)) + sqrt(Q)`,

но так, чтобы не возникало переполнений/NaN.

Сохрани `ans_07 = R` и выведи:
- долю элементов `np.isfinite(R).mean()`,
- `checksum(R)`.


In [ ]:
rng = task_rng(7)
P = rng.standard_normal(1000)
Q = rng.random(1000)

# YOUR CODE HERE

## Задание 08 — Агрегирование по осям: статистики и отбор признаков
1) Создай матрицу `Z` формы `(60, 40)` из N(0,1).
2) Для каждого столбца посчитай mean, std, median.
3) Найди индексы столбцов, где `mean > median` и `std > 0.9`.
4) Верни отсортированный список индексов и сам подмассив этих столбцов.

Сохрани:
`idx_08` (1D ndarray),
`sub_08` (2D ndarray).


In [ ]:
rng = task_rng(8)
Z = rng.standard_normal((60, 40))

# YOUR CODE HERE


## Задание 09 — Top-k по строкам через argpartition
1) Создай матрицу `W` формы `(30, 50)` из U(0,1).
2) Для каждой строки найди индексы **5 наибольших** элементов.
3) Верни индексы и значения **в порядке убывания** по каждой строке.

Сохрани:
`top_idx_09` shape `(30, 5)` (int),
`top_val_09` shape `(30, 5)` (float).


In [ ]:
rng = task_rng(9)
W = rng.random((30, 50))

# YOUR CODE HERE


## Задание 10 — Cumsum и порог для каждого ряда
1) Создай матрицу `S` формы `(20, 25)` из положительных чисел (например, exp(N(0,1))).
2) Для каждой строки найди минимальный индекс `j` (0-based), такой что:
`S[i, :j+1].sum() >= 0.9 * S[i, :].sum()`.
3) Верни вектор индексов длины 20.

Ограничения: без циклов `for/while` (разрешены cumsum, сравнения, argmax).

Сохрани `ans_10` (shape (20,)).


In [ ]:
rng = task_rng(10)
S = np.exp(rng.standard_normal((20, 25)))

# YOUR CODE HERE
